In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

from lightgbm import LGBMRegressor

from catboost import CatBoostRegressor

In [ ]:
from pathlib import Path
import os

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_ROOT = Path(os.environ.get("GANGA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("GANGA_OUTPUT_ROOT", PROJECT_ROOT / "outputs")).expanduser()

PROJECT_DIR = PROJECT_ROOT
OUTPUT_DIR = OUTPUT_ROOT / "cwc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ML_READY_DATASET = OUTPUT_DIR / "ML_Ready_Dataset.csv"
df = pd.read_csv(ML_READY_DATASET)
print(df.shape)
df.head()


In [ ]:
TARGET = "Total Coliform (MPN/100mL)"

In [ ]:
#Features
X = df.drop(columns=[TARGET])

y = df[TARGET]

print(X.shape)
print(y.shape)

In [ ]:
#Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
#Random Forest
rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train,
    y_train
)

In [ ]:
pred = rf.predict(X_test)

In [ ]:
#Evaluation
mae = mean_absolute_error(
    y_test,
    pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred
    )
)

r2 = r2_score(
    y_test,
    pred
)

print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance.head(20)

Clean Non-ML Columns

rndom forest done... 

In [ ]:
drop_extra = [
    "SlNo",
    "SlNo_x",
    "SlNo_y",
    "_id",
    "_id_x",
    "_id_y"
]

df.drop(
    columns=drop_extra,
    inplace=True,
    errors="ignore"
)

In [ ]:
TARGET = "Total Coliform (MPN/100mL)"

X = df.drop(columns=[TARGET])

y = df[TARGET]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
import re

def clean_columns(df):
    """Sanitize feature names and make them unique for estimators such as LightGBM."""
    df = df.copy()
    cleaned = []
    used = set()
    for column in df.columns:
        base = re.sub(r"[^A-Za-z0-9_]", "_", str(column))
        base = re.sub(r"_+", "_", base).strip("_") or "feature"
        candidate = base
        suffix = 1
        while candidate in used:
            candidate = f"{base}_{suffix}"
            suffix += 1
        used.add(candidate)
        cleaned.append(candidate)
    df.columns = cleaned
    return df

X_train = clean_columns(X_train)
X_test = clean_columns(X_test)


In [ ]:
print(X_train.columns[:20])

In [ ]:
models = {

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=300,
        random_state=42
    ),

    "CatBoost": CatBoostRegressor(
        iterations=300,
        random_state=42,
        verbose=False
    )

}

In [ ]:
results = []

for name, model in models.items():

    print("=" * 60)
    print(name)

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred
        )
    )

    r2 = r2_score(
        y_test,
        pred
    )

    results.append({

        "Model": name,

        "MAE": mae,

        "RMSE": rmse,

        "R2": r2

    })

    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2  :", r2)

In [ ]:
results_df = (
    pd.DataFrame(results)
    .drop_duplicates(subset="Model", keep="last")
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)

display(results_df)

In [ ]:
#Feature Importance
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

display(importance.head(30))

In [ ]:
import matplotlib.pyplot as plt

top = importance.head(20)

plt.figure(figsize=(10,8))

plt.barh(
    top["Feature"],
    top["Importance"]
)

plt.gca().invert_yaxis()

plt.title("Top 20 Important Features")
plt.tight_layout()

plt.show()